# Chunk 07 — GATv2 Training (25×25 Base Model)

Train the `AntennaGNN` GATv2 surrogate model on the **25×25 grid dataset**
(~100k samples). The 35×35, 45×45, and 55×55 grids are reserved for
transfer learning and active learning in later chunks.

**Disconnect-safe design:**
- Fixed W&B `run_id` merges metrics across reconnections.
- Checkpoint on Drive resumes epoch, optimizer, scheduler, and best val loss.
- Processed graphs are unzipped from the repo to local `/content/` disk to
  avoid Drive-FUSE latency during DataLoader iteration.
- All persistent outputs (checkpoints, figures) write to `DATA_ROOT` on Drive.

---
## Cell 1 — Install Dependencies

Install all required packages. PyG compiled extensions (`pyg-lib`, `torch-scatter`,
`torch-sparse`) are installed from the matching CUDA wheel URL.

In [1]:
!pip install scipy numpy matplotlib torch torchvision \
    torch-geometric umap-learn wandb networkx tqdm -q

# Install PyG compiled extensions matching the current CUDA version
import torch
torch_v = torch.__version__.split('+')[0]
cuda_v = 'cu' + torch.version.cuda.replace('.', '')
wheel_url = f'https://data.pyg.org/whl/torch-{torch_v}+{cuda_v}.html'
!pip install pyg-lib torch-scatter torch-sparse -f {wheel_url} -q
print(f'PyG extensions installed for torch {torch_v} + {cuda_v}')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 81.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 130.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 161.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 124.4 MB/s eta 0:00:00
PyG extensions installed for torch 2.11.0 + cu128


---
## Cell 2 — Clone Repo & Add to Path

Re-clone the GitHub repo every session (Colab wipes `/content/` on disconnect).
Add `REPO_ROOT/src` to `sys.path` so `from model import AntennaGNN` works.

In [2]:
import os
REPO_ROOT = '/content/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')   # makes 'from model import AntennaGNN' work
print(f'Repo ready at {REPO_ROOT}')

Cloning into '/content/antenna-gnn'...
remote: Enumerating objects: 125, done.
remote: Counting objects: 100% (125/125), done.
remote: Compressing objects: 100% (74/74), done.
remote: Total 125 (delta 59), reused 107 (delta 43), pack-reused 0 (from 0)
Receiving objects: 100% (125/125), 93.53 KiB | 18.71 MiB/s, done.
Resolving deltas: 100% (59/59), done.
Repo ready at /content/antenna-gnn


---
## Cell 3 — Mount Drive, Set Paths & Device Check

Mount Google Drive and configure the three standard path variables.
Assert that a GPU is available — training on CPU is not feasible.

In [3]:
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = '/content/drive/MyDrive/antenna_gnn'
RAW_DATA  = '/content/drive/MyDrive/antenna_dataset'
for d in [f'{DATA_ROOT}/artifacts', f'{DATA_ROOT}/checkpoints',
          f'{DATA_ROOT}/figures',   f'{DATA_ROOT}/splits',
          f'{DATA_ROOT}/data/processed']:
    os.makedirs(d, exist_ok=True)
print(f'Drive mounted. DATA_ROOT={DATA_ROOT}')

# --- Import model ---
from model import AntennaGNN
print('AntennaGNN imported successfully ✓')

# --- Device check ---
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'GPU not available. Go to Runtime → Change runtime type → T4 GPU'
print(f'Device: {device} ({torch.cuda.get_device_name(0)})')
print(f'GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

Mounted at /content/drive
Drive mounted. DATA_ROOT=/content/drive/MyDrive/antenna_gnn
AntennaGNN imported successfully ✓
Device: cuda (NVIDIA A100-SXM4-40GB)
GPU memory: 42.4 GB


---
## Cell 4 — Hyperparameter Config & W&B Initialization

All hyperparameters live in a single `config` dict so they are easy to audit
and automatically logged to W&B. The fixed `run_id` ensures that if the Colab
session disconnects and you re-run, W&B merges the old and new metrics into
one continuous run.

In [4]:
# ── Hyperparameters ──
config = {
    'lr':             1e-3,
    'weight_decay':   1e-5,
    'batch_size':     32,
    'hidden_dim':     128,
    'heads':          8,
    'edge_dim':       16,
    'num_blocks':     4,
    'output_dim':     201,
    'max_epochs':     200,
    'patience':       20,
    'grad_clip_norm': 1.0,
}
print('Hyperparameter config:')
for k, v in config.items():
    print(f'  {k}: {v}')

# ── W&B ──
run_id = 'antenna-gnn-v1'
use_wandb = True

try:
    import wandb
    wandb.login()
    run = wandb.init(
        project='antenna-gnn',
        config=config,
        resume='allow',
        id=run_id
    )
    print(f'W&B run: {run.url}')
except Exception as e:
    print(f'W&B login failed ({e}). Falling back to local logging.')
    use_wandb = False
    local_log = []  # fallback: list of dicts

Hyperparameter config:
  lr: 0.001
  weight_decay: 1e-05
  batch_size: 32
  hidden_dim: 128
  heads: 8
  edge_dim: 16
  num_blocks: 4
  output_dim: 201
  max_epochs: 200
  patience: 20
  grad_clip_norm: 1.0


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: nufel1130 (nufel1130-university-of-dhaka) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B run: https://wandb.ai/nufel1130-university-of-dhaka/antenna-gnn/runs/antenna-gnn-v1


---
## Cell 5 — Dataset & DataLoader (25×25 Only)

### Unzip → Local SSD

The processed 25×25 PyG graphs (~100k `.pt` files) are stored as a zip archive
at `REPO_ROOT/data/processed/processed_25x25.zip`. We unzip directly to the
fast local SSD at `/content/local_graphs/25x25/`. This avoids both:
- The Drive-FUSE latency wall (~30 ms/file × 100k = 50 min/epoch)
- Needing to manually upload the processed graphs to Drive

The unzip is a one-time cost per session (~1-2 min). A `_CACHED.txt` marker
prevents re-extraction if the cell is re-run.

### Dataset & Split Loading

We load `indices.json` (produced by Chunk 05) and **filter to 25×25 entries
only**. The 35×35, 45×45, 55×55 grids are reserved for transfer learning
and active learning in later chunks.

### num_workers=0

DataLoaders use `num_workers=0` (main-process loading) because Colab's
IPython kernel triggers spurious `AssertionError: can only test a child process`
when multiprocessing workers are garbage-collected. Since data lives on the
local SSD, single-process I/O is fast enough — GPU compute dominates.

In [5]:
import json
import shutil
import time
import zipfile
import numpy as np
from torch_geometric.data import Dataset
from torch_geometric.loader import DataLoader
from tqdm.auto import tqdm

# ═══════════════════════════════════════════════════════════════════════
# Step 1: Unzip processed 25x25 graphs to local SSD
# ═══════════════════════════════════════════════════════════════════════
LOCAL_GRAPH_ROOT = '/content/local_graphs'
N = 25
dst_dir = f'{LOCAL_GRAPH_ROOT}/{N}x{N}'
done_marker = f'{dst_dir}/_CACHED.txt'

os.makedirs(dst_dir, exist_ok=True)

if os.path.exists(done_marker):
    n_cached = len([f for f in os.listdir(dst_dir) if f.endswith('.pt')])
    print(f'Grid {N}x{N}: already cached locally ({n_cached:,} files). Skipping.')
else:
    # Locate the zip file in DATA_ROOT (Google Drive)
    zip_path = f'{DATA_ROOT}/data/processed/processed_{N}x{N}.zip'
    if not os.path.isfile(zip_path):
        raise FileNotFoundError(
            f'Zip not found at {zip_path}.\n'
            f'Please ensure the file is in your Drive at: antenna_gnn/data/processed/processed_25x25.zip'
        )

    print(f'Unzipping {zip_path} → {dst_dir} ...')
    t0 = time.time()

    with zipfile.ZipFile(zip_path, 'r') as zf:
        pt_members = [m for m in zf.namelist() if m.endswith('.pt')]
        print(f'  Found {len(pt_members):,} .pt files in archive')

        for member in tqdm(pt_members, desc=f'{N}x{N} unzip', unit='file'):
            basename = os.path.basename(member)
            target_path = os.path.join(dst_dir, basename)
            with zf.open(member) as src, open(target_path, 'wb') as dst:
                shutil.copyfileobj(src, dst)

    elapsed = time.time() - t0
    n_extracted = len([f for f in os.listdir(dst_dir) if f.endswith('.pt')])
    print(f'  Done in {elapsed:.1f}s — {n_extracted:,} files extracted')

    if n_extracted == 0:
        raise RuntimeError(f'Zip extracted 0 .pt files. Check zip contents.')

    with open(done_marker, 'w') as fh:
        fh.write('DONE\n')

print(f'\nLocal graph cache ready at {dst_dir}')

# ═══════════════════════════════════════════════════════════════════════
# Step 2: Copy artifacts (normalization stats) to local disk
# ═══════════════════════════════════════════════════════════════════════
LOCAL_ARTIFACTS = '/content/local_artifacts'
os.makedirs(LOCAL_ARTIFACTS, exist_ok=True)

for fname in ['s11_mean.npy', 's11_std.npy']:
    src = f'{DATA_ROOT}/artifacts/{fname}'
    dst = f'{LOCAL_ARTIFACTS}/{fname}'
    if not os.path.exists(dst):
        shutil.copy2(src, dst)
        print(f'Copied {fname} to local disk')

s11_mean = torch.tensor(np.load(f'{LOCAL_ARTIFACTS}/s11_mean.npy'))
s11_std  = torch.tensor(np.load(f'{LOCAL_ARTIFACTS}/s11_std.npy'))
print(f'Normalization stats loaded: mean shape={s11_mean.shape}, std shape={s11_std.shape}')

# ═══════════════════════════════════════════════════════════════════════
# Step 3: Dataset classes (read from LOCAL disk, 25x25 only)
# ═══════════════════════════════════════════════════════════════════════
class LocalGraphDataset(Dataset):
    def __init__(self, grid_size, s11_mean, s11_std):
        self.proc_dir = f'{LOCAL_GRAPH_ROOT}/{grid_size}x{grid_size}'
        self.s11_mean = s11_mean
        self.s11_std = s11_std
        self._len = len([f for f in os.listdir(self.proc_dir) if f.endswith('.pt')])
        super().__init__(root=None)

    def len(self):
        return self._len

    def get(self, idx):
        data = torch.load(f'{self.proc_dir}/sample_{idx}.pt', weights_only=False)
        if self.s11_mean is not None:
            data.y = (data.y - self.s11_mean) / (self.s11_std + 1e-8)
        return data

class SubsetDataset(Dataset):
    def __init__(self, split_indices, s11_mean, s11_std):
        self.split_indices = split_indices
        self.dataset_25 = LocalGraphDataset(25, s11_mean, s11_std)
        super().__init__(root=None)

    def len(self):
        return len(self.split_indices)

    def get(self, idx):
        _, sample_idx = self.split_indices[idx]
        return self.dataset_25.get(sample_idx)

# ═══════════════════════════════════════════════════════════════════════
# Step 4: Load split indices, filter to 25x25, create DataLoaders
# ═══════════════════════════════════════════════════════════════════════
with open(f'{DATA_ROOT}/splits/indices.json', 'r') as f:
    splits_all = json.load(f)

splits = {k: [e for e in v if e[0] == 25] for k, v in splits_all.items()}

print(f'\nFiltered to 25x25 only:')
for split_name in ['train', 'val', 'test']:
    print(f'  {split_name}: {len(splits[split_name]):,} samples')

train_dataset = SubsetDataset(splits['train'], s11_mean, s11_std)
val_dataset   = SubsetDataset(splits['val'],   s11_mean, s11_std)

train_loader = DataLoader(train_dataset, batch_size=config['batch_size'],
                          shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=config['batch_size'],
                          shuffle=False, num_workers=0, pin_memory=True)

print(f'\nTrain size: {len(train_dataset):,}')
print(f'Val size:   {len(val_dataset):,}')

Unzipping /content/drive/MyDrive/antenna_gnn/data/processed/processed_25x25.zip → /content/local_graphs/25x25 ...
  Found 99,833 .pt files in archive


25x25 unzip:   0%|          | 0/99833 [00:00<?, ?file/s]

  Done in 50.3s — 99,833 files extracted

Local graph cache ready at /content/local_graphs/25x25
Copied s11_mean.npy to local disk
Copied s11_std.npy to local disk
Normalization stats loaded: mean shape=torch.Size([201]), std shape=torch.Size([201])

Filtered to 25x25 only:
  train: 79,866 samples
  val: 9,984 samples
  test: 9,983 samples

Train size: 79,866
Val size:   9,984


---
## Cell 6 — Model, Optimizer & Scheduler

Instantiate the `AntennaGNN` model from `REPO_ROOT/src/model.py` with the
hyperparameters from `config`. Use Adam with weight decay and cosine annealing
learning rate schedule over `max_epochs`.

In [6]:
import torch.nn as nn

model = AntennaGNN(
    hidden_dim=config['hidden_dim'],
    heads=config['heads'],
    edge_dim=config['edge_dim'],
    num_blocks=config['num_blocks'],
    output_dim=config['output_dim'],
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'AntennaGNN on {device} — {total_params:,} trainable parameters')

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config['lr'],
    weight_decay=config['weight_decay']
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config['max_epochs']
)

print(f'Optimizer: Adam (lr={config["lr"]}, wd={config["weight_decay"]})')
print(f'Scheduler: CosineAnnealingLR (T_max={config["max_epochs"]})')

AntennaGNN on cuda — 587,001 trainable parameters
Optimizer: Adam (lr=0.001, wd=1e-05)
Scheduler: CosineAnnealingLR (T_max=200)


---
## Cell 7 — Checkpoint Resume

If a checkpoint exists on Drive from a previous session, load model weights,
optimizer state, scheduler state, and resume from the next epoch. This is the
core disconnect-safety mechanism.

In [7]:
CKPT_PATH = f'{DATA_ROOT}/checkpoints/best_model.pt'
start_epoch = 0
best_val_loss = float('inf')
epochs_without_improvement = 0

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    scheduler.load_state_dict(ckpt['scheduler_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['val_loss']
    print(f'✓ Resumed from epoch {start_epoch}, best val_loss={best_val_loss:.6f}')
else:
    print('Starting fresh training (no checkpoint found)')

✓ Resumed from epoch 41, best val_loss=0.688690


---
## Cell 8 — Training Loop

Main training loop with:
- **Progress bars**: per-epoch tqdm bars with loss and ETA for both train and val
- **Gradient clipping** at `config['grad_clip_norm']`
- **Gradient norm logging** (computed before clipping)
- **GPU memory cleanup** (`del batch; torch.cuda.empty_cache()`) per batch
- **Early stopping** after `patience` epochs without val loss improvement
- **Checkpoint save** to Drive on every improvement
- **W&B logging** of train_loss, val_loss, lr, grad_norm per epoch

In [8]:
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Checkpoint save function ──
def save_checkpoint(model, optimizer, scheduler, epoch, val_loss, path):
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'val_loss': val_loss,
    }, path)

# ── History tracking ──
train_losses = []
val_losses = []
best_epoch = start_epoch

total_epochs = config['max_epochs']
remaining_epochs = total_epochs - start_epoch
training_start_time = time.time()

print(f'\n{"="*70}')
print(f'Training: epochs {start_epoch} → {total_epochs-1} '
      f'({remaining_epochs} epochs remaining)')
print(f'Train batches/epoch: {len(train_loader):,} | '
      f'Val batches/epoch: {len(val_loader):,}')
print(f'{"="*70}\n')

for epoch in range(start_epoch, total_epochs):
    epoch_start = time.time()

    # ── Training pass ──
    model.train()
    running_loss = 0.0
    n_train_batches = 0
    epoch_grad_norm = 0.0
    grad_norm_count = 0

    train_pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{total_epochs-1} [train]',
                      leave=False, unit='batch')
    for batch in train_pbar:
        batch = batch.to(device)
        optimizer.zero_grad()
        out = model(batch)
        loss = criterion(out, batch.y.squeeze(1))
        loss.backward()

        # Compute gradient norm BEFORE clipping
        total_norm = 0.0
        for p in model.parameters():
            if p.grad is not None:
                total_norm += p.grad.data.norm(2).item() ** 2
        total_norm = total_norm ** 0.5
        epoch_grad_norm += total_norm
        grad_norm_count += 1

        torch.nn.utils.clip_grad_norm_(model.parameters(),
                                       max_norm=config['grad_clip_norm'])
        optimizer.step()

        running_loss += loss.item()
        n_train_batches += 1
        train_pbar.set_postfix({'loss': f'{running_loss/n_train_batches:.4f}'})

        del batch, out, loss  # free GPU memory
        torch.cuda.empty_cache()

    train_loss = running_loss / n_train_batches
    avg_grad_norm = epoch_grad_norm / max(grad_norm_count, 1)

    # ── Validation pass ──
    model.eval()
    val_running_loss = 0.0
    n_val_batches = 0

    val_pbar = tqdm(val_loader, desc=f'Epoch {epoch}/{total_epochs-1} [val]',
                    leave=False, unit='batch')
    with torch.no_grad():
        for batch in val_pbar:
            batch = batch.to(device)
            out = model(batch)
            loss = criterion(out, batch.y.squeeze(1))
            val_running_loss += loss.item()
            n_val_batches += 1
            val_pbar.set_postfix({'val_loss': f'{val_running_loss/n_val_batches:.4f}'})

            del batch, out, loss
            torch.cuda.empty_cache()

    val_loss = val_running_loss / n_val_batches

    # ── Scheduler step ──
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]

    # ── Logging ──
    train_losses.append(train_loss)
    val_losses.append(val_loss)

    log_dict = {
        'train_loss': train_loss,
        'val_loss': val_loss,
        'lr': current_lr,
        'grad_norm': avg_grad_norm,
        'epoch': epoch,
    }

    if use_wandb:
        wandb.log(log_dict, step=epoch)
    else:
        local_log.append(log_dict)

    # ── Checkpoint & early stopping ──
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        save_checkpoint(model, optimizer, scheduler, epoch, val_loss, CKPT_PATH)
    else:
        epochs_without_improvement += 1

    # ── Timing & ETA ──
    epoch_time = time.time() - epoch_start
    elapsed_total = time.time() - training_start_time
    epochs_done = epoch - start_epoch + 1
    avg_epoch_time = elapsed_total / epochs_done
    eta_seconds = avg_epoch_time * (total_epochs - epoch - 1)
    eta_min = eta_seconds / 60
    pct_complete = 100.0 * epochs_done / remaining_epochs

    # ── Print progress every 5 epochs (or first/last) ──
    if epoch % 5 == 0 or epoch == total_epochs - 1 or epochs_without_improvement >= config['patience']:
        print(f'Epoch {epoch}/{total_epochs-1} | '
              f'train: {train_loss:.4f} | val: {val_loss:.4f} | '
              f'lr: {current_lr:.2e} | grad_norm: {avg_grad_norm:.2f} | '
              f'{epoch_time:.1f}s/epoch | '
              f'{pct_complete:.0f}% done | ETA: {eta_min:.1f} min')
        if val_loss >= best_val_loss and epochs_without_improvement > 0:
            print(f'  └─ no improvement for {epochs_without_improvement}/{config["patience"]} epochs '
                  f'(best: {best_val_loss:.4f} @ epoch {best_epoch})')

    # ── Early stopping ──
    if epochs_without_improvement >= config['patience']:
        print(f'\n⚠ Early stopping at epoch {epoch} '
              f'(no improvement for {config["patience"]} epochs)')
        break

total_training_time = time.time() - training_start_time
print(f'\n{"="*70}')
print(f'Training complete!')
print(f'  Best epoch:     {best_epoch}')
print(f'  Best val loss:  {best_val_loss:.6f}')
print(f'  Total time:     {total_training_time/60:.1f} min')
print(f'{"="*70}')


Training: epochs 41 → 199 (159 epochs remaining)
Train batches/epoch: 2,496 | Val batches/epoch: 312



Epoch 41/199 [train]:   0%|          | 0/2496 [00:00<?, ?batch/s]

Epoch 41/199 [val]:   0%|          | 0/312 [00:00<?, ?batch/s]

Epoch 42/199 [train]:   0%|          | 0/2496 [00:00<?, ?batch/s]

Epoch 42/199 [val]:   0%|          | 0/312 [00:00<?, ?batch/s]

...

Epoch 59/199 [train]:   0%|          | 0/2496 [00:00<?, ?batch/s]

Epoch 59/199 [val]:   0%|          | 0/312 [00:00<?, ?batch/s]

Epoch 60/199 [train]:   0%|          | 0/2496 [00:00<?, ?batch/s]

Epoch 60/199 [val]:   0%|          | 0/312 [00:00<?, ?batch/s]

Epoch 60/199 | train: 0.3948 | val: 0.7020 | lr: 7.88e-04 | grad_norm: 0.66 | 236.1s/epoch | 13% done | ETA: 549.1 min
  └─ no improvement for 20/20 epochs (best: 0.6887 @ epoch 41)

⚠ Early stopping at epoch 60 (no improvement for 20 epochs)

Training complete!
  Best epoch:     41
  Best val loss:  0.688690
  Total time:     79.0 min


---
## Cell 9 — Training Curves

Plot train and validation loss vs epoch. A vertical dashed line marks the
best epoch (lowest validation loss). The figure is saved to Drive at 300 DPI
and also logged to W&B.

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

epochs_range = list(range(start_epoch, start_epoch + len(train_losses)))

ax.plot(epochs_range, train_losses, label='Train Loss', color='#2196F3',
        linewidth=2, alpha=0.85)
ax.plot(epochs_range, val_losses, label='Val Loss', color='#FF5722',
        linewidth=2, alpha=0.85)
ax.axvline(x=best_epoch, color='#4CAF50', linestyle='--', linewidth=1.5,
           label=f'Best epoch ({best_epoch})', alpha=0.8)

ax.set_xlabel('Epoch', fontsize=13)
ax.set_ylabel('MSE Loss (normalized S11)', fontsize=13)
ax.set_title('AntennaGNN Training Curves (25×25 Base)', fontsize=15, fontweight='bold')
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)
ax.set_xlim(epochs_range[0], epochs_range[-1])

# Annotate best val loss
best_idx = best_epoch - start_epoch
if 0 <= best_idx < len(val_losses):
    ax.annotate(f'Val: {val_losses[best_idx]:.4f}',
                xy=(best_epoch, val_losses[best_idx]),
                xytext=(best_epoch + 5, val_losses[best_idx] * 1.1),
                arrowprops=dict(arrowstyle='->', color='#4CAF50'),
                fontsize=10, color='#4CAF50', fontweight='bold')

plt.tight_layout()
fig_path = f'{DATA_ROOT}/figures/training_curves.png'
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()
print(f'Figure saved to {fig_path}')

# Log figure to W&B
if use_wandb:
    wandb.log({'training_curves': wandb.Image(fig_path)})
    print('Figure logged to W&B')

# ── Final summary ──
print(f'\n{"="*50}')
print(f'Final Statistics')
print(f'  Best epoch:            {best_epoch}')
print(f'  Best val loss:         {best_val_loss:.6f}')
print(f'  Total training time:   {total_training_time/60:.1f} min')
print(f'  Epochs trained:        {len(train_losses)}')
print(f'  Checkpoint saved at:   {CKPT_PATH}')
print(f'{"="*50}')

if use_wandb:
    wandb.finish()
    print('W&B run finished.')

Figure saved to /content/drive/MyDrive/antenna_gnn/figures/training_curves.png
Figure logged to W&B

Final Statistics
  Best epoch:            41
  Best val loss:         0.688690
  Total training time:   79.0 min
  Epochs trained:        20
  Checkpoint saved at:   /content/drive/MyDrive/antenna_gnn/checkpoints/best_model.pt


epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
grad_norm,▁▂▂▂▃▃▃▃▄▄▅▅▅▆▆▆▇▇▇█
lr,██▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▁▁
train_loss,██▇▇▇▆▆▅▅▅▄▄▄▃▃▂▂▂▁▁
val_loss,▃▂▁▂▃▂▂▁▂▂▄▄▃▆▂▄▁▆▅█
epoch,60
grad_norm,0.6562
lr,0.00079
train_loss,0.39483
val_loss,0.702


W&B run finished.
